# Building the Experiment 1b (proportion) stimuli

This notebook writes `materials/src/proportion-items.json`, the stimulus file
`proportion-experiment.js` loads at run time.

Experiment 1b reuses Experiment 1a's sentences, but where 1a let participants
*choose* a temporal modifier, 1b fixes one and asks about it. The central question is which modifier to fix. Following §3.1 of the paper, we give each sentence the modifier that participants produced for it most often in the cloze task,
computed separately under *in* and under *for*, since the two prepositions tend to
attract rather different durations even for the same sentence.

The notebook proceeds in four steps:

1. read the Experiment 1a responses and keep the episodic ones;
2. for each (sentence, preposition), find the modal time unit and then the
   modal numeral within it;
3. build the prompt and the five response options around that modifier,
   inflecting for number;
4. distribute the resulting items across lists, balanced by preposition.

Paths are relative to this notebook's directory. Run it from
`materials/scripts/`, or use `jupyter nbconvert --execute`, which sets that
working directory automatically.

In [ ]:
import pandas as pd

from scipy.stats import skew, kurtosis

## Load the Experiment 1a responses

We derive the modifiers from cloze responses, so this reads
`data/data-cloze.csv` rather than the sentence inventory.

We drop rows with a missing numeral and then restrict to the episodic
clause type. The `astype(int).astype(str)` round trip normalizes the numeral to a bare
string, since we use it as a dictionary key and in the item ID below.

In [ ]:
data = pd.read_csv("../../data/data-cloze.csv")
# Proportion items are derived from the episodic cloze data.
data = data[data["numeral"].notna()].copy()
data["numeral"] = data["numeral"].astype(int).astype(str)

data_episodic = data.query('clause_type == "episodic"')

data_episodic

## Number agreement for the time unit

The cloze task recorded time units in the plural (*minutes*, *hours*), but a fixed modifier with the numeral 1 needs the singular (*in 1 minute*, not
*in 1 minutes*). This map lets us inflect whenever the modal numeral turns out to be 1.

The plural list is also in ascending order of duration, which the `duration_index` column in the next cell relies on.

In [ ]:
duration_plural = [
    "milliseconds",
    "seconds",
    "minutes",
    "hours",
    "days",
    "weeks",
    "months",
    "years",
    "decades",
    "centuries",
    "millenia"
]

duration_singular = [
    "millisecond",
    "second",
    "minute",
    "hour",
    "day",
    "week",
    "month",
    "year",
    "decade",
    "century",
    "millenium"
]

duration_map = dict(zip(duration_plural, duration_singular))

## Progressive forms for the question

The prompt asks about the event as ongoing (*For what proportion of those five
minutes was the child hitting the pillow?*), so the progressive participle of
each of the 40 verbs is needed. These are written out by hand rather than generated, since English participle
spelling is irregular enough (consonant doubling in *hitting* and *stabbing*,
*e*-deletion in *wiping* and *deciding*) that a rule would need about as many
exceptions as entries.

In [ ]:
progressive_map = {
	"hit": "hitting",
	"wipe": "wiping",
	"stab": "stabbing",
	"shoot": "shooting",
	"ponder": "pondering",
	"kick": "kicking",
	"fight": "fighting",
	"push": "pushing",
	"investigate": "investigating",
	"wash": "washing",
	"bump": "bumping",
	"pull": "pulling",
	"debate": "debating",
	"jab": "jabbing",
	"pour": "pouring",
	"smear": "smearing",
	"scrub": "scrubbing",
	"pound": "pounding",
	"shake": "shaking",
	"consider": "considering",
	"break": "breaking",
	"clean": "cleaning",
	"kill": "killing",
	"murder": "murdering",
	"destroy": "destroying",
	"open": "opening",
	"fold": "folding",
	"conclude": "concluding",
	"discover": "discovering",
	"decide": "deciding",
	"empty": "emptying",
	"chip": "chipping",
	"close": "closing",
	"prove": "proving",
	"shatter": "shattering",
	"cover": "covering",
	"split": "splitting",
	"fill": "filling",
	"clear": "clearing",
	"combine": "combining"
}

## Construct the items

For each sentence and preposition, we select a modifier and assemble the prompt
around it.

**Selecting the modifier.** We group by sentence and take the responses given
under the preposition in question, then read off the modal time unit and, within
that time unit, the modal numeral. Where a sentence never received a given preposition in the cloze task, we fall
back to its responses under the other one, so that every sentence still gets an item in both preposition conditions. That
fallback is rare, but it does mean that a small number of items carry a modifier normed
under the opposite preposition.

**Assembling the prompt.** Three details matter:

- *Number agreement.* The determiner is *that* for a numeral of 1 and *those N*
  otherwise, and we inflect the time unit to match through `duration_map`.
- *The progressive clause.* We rebuild the sentence with *likely* plus the
  participle in place of the past-tense verb, keeping the original arguments.
  Templatic sentences are offset by one token because their subject is the bare
  variable *X* rather than a determiner-noun sequence.
- *The auxiliary.* This has to agree with the subject, which is why we carry `subject_noun_plural` all the way down from the grouping key.

**Deduplication.** Bleached sentences are shared across senses, so the same
sentence can arrive several times from different rows. We track which sentences
have already been emitted per (contentfulness, preposition) and keep only the
first, which is why the item count is well below the number of input rows.

In [ ]:
data_episodic["duration_index"] = data_episodic.duration.map(duration_plural.index)

items = []

sentences_seen = {
    "contentful": {"in": [], "for": []},
    "bleached": {"in": [], "for": []},
    "templatic": {"in": [], "for": []}
}

for (contentfulness, verb_type, verb, sentence, subject_noun_plural), df_sent in data_episodic.groupby(["contentfulness", "verb_type", "verb", "sentence", "subject_noun_plural"]):
    for preposition in ["in", "for"]:
        df_prep = df_sent[df_sent.preposition == preposition]

        if df_prep.shape[0] == 0:
            durations = df_sent.value_counts("duration").sort_values(ascending=False)
        else:
            durations = df_prep.value_counts("duration").sort_values(ascending=False)
        
        durations = durations.index[:1]
        #durations = durations.index[:2]
        #durations = durations[durations > 1].index[:2]

        for d in durations:
            if df_prep.shape[0] == 0:
                numeral = df_sent[df_sent.duration == d].value_counts("numeral").sort_values(ascending=False).index[0]
            else:
                numeral = df_prep[df_prep.duration == d].value_counts("numeral").sort_values(ascending=False).index[0]
                
            numeral = int(numeral.replace(",", "")) if "," in numeral else int(numeral)

            determiner = "that" if numeral == 1 else f"those {numeral}"

            if contentfulness != "templatic":
                progressive_sentence = " ".join(
                    sentence.lower().split()[:2] + 
                    ["likely", progressive_map[verb]] + 
                    sentence.strip(".").split()[3:]
                )
            else:
                progressive_sentence = " ".join(
                    sentence.split()[:1] + 
                    ["likely", progressive_map[verb]] + 
                    sentence.strip(".").split()[2:]
                )

            sent = sentence[:-1]#.lower() if contentfulness != 'templatic' else sentence[:-1]
            d_infl = d if numeral != 1 else duration_map[d]

            prog_sent = (progressive_sentence.capitalize() if contentfulness != "templatic" else progressive_sentence).replace("likely", "was") + "."

            prompt = (
                f"<span class=\"sentence\"><span class=\"clause\">{sent}</span> {preposition} <span class=\"time\">{numeral} {d_infl}</span>.</span><br/><br/>"
                f"<span class=\"question\">For what proportion of <span class=\"time\">{determiner} {d_infl}</span> (if any) <span class=\"clause\">{'were' if subject_noun_plural else 'was'} {progressive_sentence}</span>?</span>"
            )

            item_id = f"{contentfulness}-{verb_type}-{verb}-{sentence}-{preposition}-{numeral}-{d}"

            options = (
                f"None of {determiner} {d_infl}",
                f"Less than half of {determiner} {d_infl}",
                f"About half of {determiner} {d_infl}",
                f"More than half of {determiner} {d_infl}",
                f"The entirety of {determiner} {d_infl}"
            )


            if sentence not in sentences_seen[contentfulness][preposition]:
                items.append([
                    item_id,
                    verb_type,
                    contentfulness, 
                    verb,
                    sentence,
                    prog_sent,
                    preposition,
                    numeral,
                    d, 
                    prompt,
                    options
                ])

                sentences_seen[contentfulness][preposition].append(sentence)

items = pd.DataFrame(
    items,
    columns=[
        "item_id", "verb_type", "contentfulness", "verb", "sentence_past", 
        "sentence_progressive", "preposition", "numeral", "duration",
        "prompt", "options"
    ]
)

items

## Assign items to lists

We split contentful items across 8 lists, bleached across 5, and templatic
across 2.

Our assignment keeps preposition balanced within each list. Each preposition
holds a queue of list indices, initialized to disjoint halves of the list set,
and whenever we assign an item we append the list it went to onto the *other*
preposition's queue, so that a list which has just received an *in* item
becomes next in line for a *for* item. Working verb by verb and alternating prepositions this
way spreads both factors across lists without requiring an explicit Latin square.

In [ ]:
lists = {
    "contentful": [[] for _ in range(8)],
    "bleached": [[] for _ in range(5)],
    "templatic": [[] for _ in range(2)]
}

next_prep = {
    clause_type: {
        "in": list(range(int(len(ls)/2))),
        "for": list(range(int(len(ls)/2), len(ls)))
    }
    for clause_type, ls in lists.items()
}

for contentfulness, df_cont in items.groupby("contentfulness"):
    alternate = False
    for verb_type, df_vt in df_cont.groupby("verb_type"):
        for verb, df_verb in df_vt.groupby("verb"):
            df_verb_parts = {
                p: [
                    r.to_dict() for _, r in df_verb[df_verb.preposition == p].iterrows()
                ] 
                for p in ["in", "for"]
            }

            items_remaining = [p for p in ["in", "for"] if df_verb_parts[p]]

            while items_remaining:
                for p in items_remaining:
                    item = df_verb_parts[p].pop()
                    list_num = next_prep[contentfulness][p].pop(0)

                    lists[contentfulness][list_num].append(item)

                    other_prep = "in" if p == "for" else "for"

                    next_prep[contentfulness][other_prep].append(list_num)

                    items_remaining = [p for p in ["in", "for"] if df_verb_parts[p]]


    print(f"Length of lists for {contentfulness}:", [len(l) for l in lists[contentfulness]])

## Check the balance

The next five cells count verb type, verb, preposition, time unit, and numeral
within each list. Balanced lists have roughly even counts for the first three
variables and are not dominated by one duration.

These diagnostics print rather than assert. Inspect their output because a
lopsided list does not raise an error.

In [ ]:
from collections import Counter

for contentfulness, ls in lists.items():
    for j, l in enumerate(ls):
        c = Counter()
        for i in l:
            c[i["verb_type"]] += 1
        
        print(contentfulness, j, c)

    print()

In [ ]:
for contentfulness, ls in lists.items():
    for j, l in enumerate(ls):
        c = Counter()
        for i in l:
            c[i["verb"]] += 1
        
        print(contentfulness, j, c)

    print()

In [ ]:
for contentfulness, ls in lists.items():
    for j, l in enumerate(ls):
        c = Counter()
        for i in l:
            c[i["preposition"]] += 1
        
        print(contentfulness, j, c)

    print()

In [ ]:
for contentfulness, ls in lists.items():
    for j, l in enumerate(ls):
        c = Counter()
        for i in l:
            c[i["duration"]] += 1
        
        print(contentfulness, j, c)

    print()

In [ ]:
for contentfulness, ls in lists.items():
    for j, l in enumerate(ls):
        c = Counter()
        for i in l:
            c[i["numeral"]] += 1
        
        print(contentfulness, j, c)

    print()

## Write the stimulus file

This cell writes the gitignored file `materials/src/proportion-items.json`,
so regeneration does not appear as a repository change.

In [ ]:
import json

with open("../src/proportion-items.json", "w") as f:
    json.dump(lists, f, indent=4)